# P2P Circuit Sharing via Knitweb Fabric
This notebook shows how to push circuits to the Knitweb relay and pull them by CID — identical to how Knitweb nodes share knowledge across the network.

**No central server.** The relay is a thin HTTPS bridge to the Knitweb P2P fabric. Any node holding a CID can serve it.


In [ ]:
!pip install -q knitweb-lens

In [ ]:
# Configure relay (replace with your Knitweb relay URL)
RELAY = 'https://relay.5mart.ml'  # or None for local-only

from knitweb_lens.store import Store
store = Store(relay=RELAY)
print(store)

In [ ]:
# Push a circuit
from knitweb_lens.adapters.qasm import from_qasm

qasm = '''
OPENQASM 2.0;
include "qelib1.inc";
// Custom VQE ansatz for H2 dissociation curve
qreg q[4];
creg c[4];
x q[0]; x q[2];
ry(0.314159) q[0];
cx q[0],q[1];
ry(-0.314159) q[0];
ry(0.314159) q[2];
cx q[2],q[3];
ry(-0.314159) q[2];
cx q[1],q[2];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
measure q[3] -> c[3];
'''

circ = from_qasm(qasm, name='vqe_h2_custom', domain='algorithms',
                 tags=['vqe','h2','chemistry'],
                 description='Custom H2 VQE ansatz at 0.742 Å bond length',
                 author='Colab demo')

cid = store.put(circ)
print(f'Pushed! CID: {cid}')
print(f'Share this CID with anyone on the Knitweb network.')

In [ ]:
# Pull by CID — this is what another node would do
retrieved = store.get(cid)
print(f'Retrieved: {retrieved.meta.name}')
print(f'Author   : {retrieved.meta.author}')
print(f'Tags     : {retrieved.meta.tags}')
print()
print(retrieved.qasm)

In [ ]:
# Browse the full built-in library via the store
from knitweb_lens import library

# Pre-populate local store with all 100 built-in circuits
lib = library()
for name, circ in lib.items():
    store.put(circ)

print(f'Store now contains {len(store)} circuits')

# Search locally
hits = store.find(query='ising')
print(f'\nIsing circuits:')
for h in hits:
    print(f"  {h['meta']['name']:<30} {h['cid']}")

In [ ]:
# Verify content integrity via CID
from knitweb_lens.cid import verify

circ2 = store.get(cid)
ok = verify(circ2.qasm, cid)
print(f'CID verified: {ok}')  # True = bit-perfect

# Tamper detection
tampered = circ2.qasm + '\n// malicious injection'
print(f'Tampered verifies: {verify(tampered, cid)}')  # False